# BF1 — Knowledge Graph Explorer

Explores the BIRD museum knowledge graph structure: entity counts, relationship types,
artwork/trajectory overview. Uses `TrajectoNXGraphBuilder` for graph construction.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('helpers'))
from fuseki_config import query_endpoint
from fuseki_queries import execute_sparql_query, format_results
from graph_builder import TrajectoNXGraphBuilder

In [ ]:
PREFIXES = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi:      <http://w3id.org/trajecto/poi#>
PREFIX patterns: <http://w3id.org/trajecto/patterns#>
PREFIX purpose:  <http://w3id.org/trajecto/purpose#>
PREFIX user:     <http://w3id.org/trajecto/user#>
PREFIX dul:      <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX datacron: <http://www.datacron-project.eu/datAcron#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:      <http://www.w3.org/2001/XMLSchema#>
"""

## 1. Entity Counts

In [ ]:
entity_queries = [
    ('Total triples', 'SELECT (COUNT(*) AS ?c) WHERE { ?s ?p ?o }'),
    ('Trajectories', 'SELECT (COUNT(DISTINCT ?t) AS ?c) WHERE { ?t a datacron:Trajectory }'),
    ('Persons', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a trajecto:Person }'),
    ('Artworks (POIs)', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a trajecto:PointOfInterest }'),
    ('Positions', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a datacron:RawPosition }'),
    ('Stop Positions', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a trajecto:StopPosition }'),
    ('POI Interactions', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a poi:POIInteraction }'),
    ('Meaningful', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a poi:MeaningfulInteraction }'),
    ('Immersive', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a poi:ImmersiveInteraction }'),
    ('Passing', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a poi:PassingInteraction }'),
    ('Dominant', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a poi:DominantInteraction }'),
    ('Significant POIs', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a poi:SignificantPOI }'),
    ('User Profiles', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a user:UserProfile }'),
]

print('=== Knowledge Graph Entity Counts ===')
for label, query in entity_queries:
    result = execute_sparql_query(query_endpoint, PREFIXES + query)
    df = format_results(result)
    count = df.iloc[0]['c'] if not df.empty else '?'
    print(f'  {label}: {count}')

## 2. Artwork Types Distribution

In [ ]:
q = PREFIXES + """
SELECT ?type (COUNT(DISTINCT ?poi) AS ?count)
WHERE {
    ?poi a trajecto:PointOfInterest .
    ?poi poi:hasType ?type .
}
GROUP BY ?type
ORDER BY DESC(?count)
"""
result = execute_sparql_query(query_endpoint, q)
print('Artwork types in KG:')
format_results(result)

## 3. Navigation Styles & Purposes

In [ ]:
# Navigation styles
q = PREFIXES + """
SELECT ?style (COUNT(?traj) AS ?count)
WHERE {
    ?traj patterns:exhibitsPattern ?style .
}
GROUP BY ?style ORDER BY DESC(?count)
"""
result = execute_sparql_query(query_endpoint, q)
print('Navigation Styles:')
display(format_results(result))

# Purposes
q = PREFIXES + """
SELECT ?purpose (COUNT(?traj) AS ?count)
WHERE {
    ?traj trajecto:hasPurpose ?purpose .
}
GROUP BY ?purpose ORDER BY DESC(?count)
"""
result = execute_sparql_query(query_endpoint, q)
print('\nTrajectory Purposes:')
format_results(result)

## 4. Graph Builder Overview

In [ ]:
builder = TrajectoNXGraphBuilder()

# POI metadata
poi_meta = builder.get_poi_metadata()
print(f'POI metadata entries: {len(poi_meta)}')
if not poi_meta.empty:
    print(f'Columns: {list(poi_meta.columns)}')
    display(poi_meta.head(10))

In [ ]:
# Trajectory metadata
traj_meta = builder.get_trajectory_metadata()
print(f'Trajectory metadata entries: {len(traj_meta)}')
if not traj_meta.empty:
    display(traj_meta.head(10))

In [ ]:
# User metadata
user_meta = builder.get_user_metadata()
print(f'Users: {len(user_meta)}')
if not user_meta.empty:
    display(user_meta.head(10))

## 5. Sample Trajectory Artwork Sequences

In [ ]:
sequences = builder.get_trajectory_poi_sequences()
print(f'Trajectories with sequences: {len(sequences)}')
for traj, pois in list(sequences.items())[:3]:
    traj_short = traj.split('#')[-1] if '#' in traj else traj.split('/')[-1]
    print(f'\n  {traj_short}: {len(pois)} artworks')
    for poi in pois[:5]:
        poi_short = poi.split('#')[-1] if '#' in poi else poi.split('/')[-1]
        print(f'    -> {poi_short}')